In [84]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from time import sleep
import logging
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions 
from selenium.common.exceptions import TimeoutException
import random
import httpx
import lxml

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(levelname)s - %(message)s',datefmt='%H:%M:%S',force=True)

Для работы с BeautifulSoup была использована информация из источника: https://habr.com/ru/companies/wunderfund/articles/683880/ и https://tproger.ru/articles/shpargalka-po-logirovaniju-na-python

Для работы с Selenium была использована информация из источника: https://habr.com/ru/companies/otus/articles/596071/

In [85]:
def driver_init():
    driver = webdriver.Chrome()
    logging.info('Браузер Selenium успешно запущен')
    driver.get("https://store.steampowered.com/search/?ndl=1")
    logging.info('Открыта страница поиска Steam')
    return driver

In [86]:
driver = driver_init()

18:13:19 - INFO - Браузер Selenium успешно запущен
18:13:20 - INFO - Открыта страница поиска Steam


In [ ]:
def scrolldown(driver, vniz): #крч функция, которая мотает страницу вниз, чтобы больше игр прогрузилось
    logging.info('Начало прокрутки страницы')
    for i in range(vniz):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") #JavaScript-код для браузера, который обращается к DOM страницы
        #https://developer.mozilla.org/en-US/docs/Web/API/Window/scrollTo?.com
        #https://developer.mozilla.org/en-US/docs/Web/API/Document/body?.com
        #https://developer.mozilla.org/en-US/docs/Web/API/Element/scrollHeight?.com
        logging.info(f'Скролл номер {i+1}')
        sleep(random.uniform(0.5, 1.2))
    logging.info('Прокрутка завершена')

In [ ]:
scrolldown(driver, 220)

17:36:22 - INFO - Начало прокрутки страницы
17:36:22 - INFO - Скролл номер 1
17:36:25 - INFO - Скролл номер 2
17:36:27 - INFO - Скролл номер 3
17:36:29 - INFO - Скролл номер 4
17:36:32 - INFO - Прокрутка завершена


In [82]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
urls = []

for link in soup.find_all('a'):
    if '/app' in link.get('href') and link.get('href').startswith('https://'):
        urls.append(link.get('href'))
        
logging.info(f'Собрано ссылок на игры: {len(urls)}')

17:54:34 - INFO - Собрано ссылок на игры: 250


In [89]:
url0 = urls[0]

driver.get(url0)
html = driver.page_source
soup0 = BeautifulSoup(html, 'html.parser')
print(soup0.prettify())

<html class="responsive DesktopUI" lang="ru">
 <head>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <meta content="#171a21" name="theme-color"/>
  <title>
   Counter-Strike 2 в Steam
  </title>
  <link href="/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
  <link href="https://store.fastly.steamstatic.com/public/shared/css/motiva_sans.css?v=YzJgj1FjzW34&amp;l=russian&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
  <link href="https://store.fastly.steamstatic.com/public/shared/css/shared_global.css?v=qbbUOQL2SbhT&amp;l=russian&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
  <link href="https://store.fastly.steamstatic.com/public/shared/css/buttons.css?v=BZhNEtESfYSJ&amp;l=russian&amp;_cdn=fastly" rel="stylesheet" type="text/css"/>
  <link href="https://store.fastly.steamstatic.com/public/css/v6/store.css?v=13CjnPuGjLMj&amp;l=russian&amp;_cdn=fastly" rel="stylesheet" type

Для работы с BeautifulSoup была использована информация из источника: https://habr.com/ru/articles/986284/

In [106]:
#название
name = soup0.find_all('div', {'class': 'apphub_AppName'})[0].text
print(name)

Counter-Strike 2


In [97]:
#дата выпуска
release_date = soup0.find_all('div', {'class': 'date'})[0].text.strip()
print(release_date)

21 авг. 2012 г.


In [98]:
#разработчики
developers = soup0.find_all('div', {'id': 'developers_list'})[0].text.strip()
print(developers)

Valve


In [152]:
#жанр
genre = []
block = soup0.find_all('div', {'class': 'details_block'})
for i in block:
    b_all = i.find_all('b')
    for b in b_all:
        if 'Жанр:' in b.text:
            span = b.find_next('span')
            links = span.find_all('a')
            for g in links:
                genre.append(g.text.strip())
            break
print(genre)

['Экшены', 'Бесплатные']


In [64]:
#цена без скидок
try:
     price = soup0.find_all('div', {'class': 'game_purchase_price price'})[0].text.strip()
except:
     price = soup0.find_all('div', {'class': 'discount_original_price'})[0].text.strip()
print(price)

Бесплатные


In [65]:
#скидка
blocks = soup0.find_all('div', {'class': 'game_area_purchase_game_wrapper'})
discount = '0%'
for block in blocks:
    title = block.find('h2')
    title = title.text.strip()
    if title == 'Купить ' + name:
        d = block.find('div', {'class': 'discount_pct'})
        if d:
            discount = d.text.strip()
        break

print(discount)

0%


In [99]:
# количество DLC
dlc = 0
dlc_all = soup0.find('div', {'class': 'game_area_dlc_section'})
if dlc_all:
    dlc = int(dlc_all.find_all('em')[0].text.strip().replace('(', '').replace(')', ''))
print(dlc)

1


In [27]:
#теги
tags = []
for tag in soup0.find_all('a', {'class': 'app_tag'}):
    tags.append(tag.text.strip())
print(tags)

['Стратегия', 'Рогалик', 'Карточная игра', 'Построение колоды', 'Кооператив', 'Карточный рогалик', 'Для нескольких игроков', 'Пошаговые сражения', 'Карточный баттлер', 'Инди', 'Подземелья', 'Упрощённый рогалик', '2D', 'Ранний доступ', 'Для одного игрока', 'Сложная', 'Фэнтези', 'Реиграбельность', 'Процедурная генерация', 'Управление мышью']


In [125]:
#недавний рейтинг
block = soup0.find_all('div', {'class': 'outlier_totals recent review_box_background_secondary'})
for b in block:
    if b and b.find('span', {'class': 'game_review_summary'}):
        recent_rating = soup0.find_all('span', {'class': 'game_review_summary'})[0].text
print(recent_rating)

В основном положительные


In [126]:
#количество недавних отзывов
block = soup0.find_all('div', {'class': 'outlier_totals recent review_box_background_secondary'})
for b in block:
    if b and b.find('span', {'class': 'game_review_summary'}):
        recent_count = b.find('span', {'class': 'review_summary_count'}).text.strip()
print(recent_count)

94 027


In [123]:
#рейтинг
block = soup0.find_all('div', {'class': 'outlier_totals global review_box_background_secondary'})
for b in block:
    if b and b.find('span', {'class': 'game_review_summary'}):
        rating = b.find('span', {'class': 'game_review_summary'}).text.strip()
print(rating)


Очень положительные


In [124]:
#количество всех отзывов
block = soup0.find_all('div', {'class': 'outlier_totals global review_box_background_secondary'})
for b in block:
    if b and b.find('span', {'class': 'game_review_summary'}):
        count = b.find('span', {'class': 'review_summary_count'}).text.strip()
print(count)

9 517 910


Есть проблема при входе на страницы некоторых игр, где есть возрастное ограничение и поэтому не удается с них собрать данные, если не подтвердить возраст, поэтому необходимо также через библиотеку selenium нужно выбрать возраст > 18 лет

expected_conditionsh и presence_of_element_located - https://www.selenium.dev/selenium/docs/api/py/selenium_webdriver_support/selenium.webdriver.support.expected_conditions.html


In [71]:
def pass_age_gate(driver): #функция подтверждения возраста
    try:
        WebDriverWait(driver, 3).until(expected_conditions.presence_of_element_located((By.ID, "ageYear"))) # тут крч жду до 3 секунд элемент с id ageYear и если такой элемент появился, значит перед нами форма подтверждения возраста
        logging.info("Обнаружена страница с подтверждением возраста")
        
        # тут выбираю день, месяц и год рождения, чтобы указать возраст старше 18 лет
        day = Select(driver.find_element(By.ID, "ageDay"))
        month = Select(driver.find_element(By.ID, "ageMonth"))
        year = Select(driver.find_element(By.ID, "ageYear"))
        day.select_by_visible_text("27")
        month.select_by_visible_text("September")
        year.select_by_visible_text("2005")
        
        driver.find_element(By.ID, "view_product_page_btn").click() #тут нажимаю продолжить 
        WebDriverWait(driver, 5).until(expected_conditions.presence_of_element_located((By.CLASS_NAME, "apphub_AppName"))) # это команда ожидания, чтобы Selenium не начал парсить страницу слишком рано
        logging.info("Возраст подтвержден, страница игры открыта")
    except TimeoutException:
        logging.info("Страница с подтверждением возраста не обнаружена")

После того как мы первый раз запустили нашу функцию и она отрабатывала 15 часов, мы поняли, что надо что-то менять и использовали доп библиотеку httpx https://habr.com/ru/companies/ru_mts/articles/968776/

In [ ]:
timeout_config = httpx.Timeout(10.0)
client = httpx.Client(timeout=timeout_config)
def GetGame(url0):
    try:
        logging.info('Начало обработку игры')
        """
        Возвращает кортеж:
        url0, name, release_date, developers, genre, price, discount, dlc, tags, rating, count, recent_rating, recent_count, 
        где:
        - name - название игры
        - release_date - дата релиза
        - developers - разработчики
        - genre - список жанров
        - price - цена
        - discount - скидка
        - dlc - количество dlc
        - tags - теги
        - rating - общий рейтинг
        - count - общее количество отзывов
        - recent_rating - недавний рейтинг
        - recent_count - количество недавних отзывов
        """
        if 'l=english' not in url0:
            if '?' in url0:
                url0 = url0 + '&l=english'
            else:
                url0 = url0 + '?l=english'
        response = client.get(url0)
        page0 = response.text
        soup0 = BeautifulSoup(page0, 'html.parser')

        if len(soup0.find_all('div', {'class': 'apphub_AppName'})) == 0:
            logging.warning('Не удалось использовать client')
            driver.get(url0)
            pass_age_gate(driver)
            page0 = driver.page_source
            soup0 = BeautifulSoup(page0, 'html.parser')

        name = None
        release_date = None
        developers = None
        genre = None
        price = None
        discount = '0%'
        dlc = 0
        tags = None
        rating = None
        count = None
        recent_rating = None
        recent_count = None
        try:
            name = soup0.find_all('div', {'class': 'apphub_AppName'})[0].text.strip()
        except:
            name = None
            logging.warning('Не удалось найти название игры')

        try:
            release_date = pd.to_datetime(soup0.find_all('div', {'class': 'date'})[0].text.strip(), format='%d %b, %Y')
            #release_date = soup0.find_all('div', {'class': 'date'})[0].text.strip()
        except:
            release_date = None
            logging.warning('Не удалось найти дату релиза')

        try:
            developers = soup0.find_all('div', {'id': 'developers_list'})[0].text.strip()
        except:
            developers = None
            logging.warning('Не удалось найти разработчика')

        try:
            genre = []
            block = soup0.find_all('div', {'class': 'details_block'})
            for i in block:
                b_all = i.find_all('b')
                for b in b_all:
                    if ('Жанр' in b.text) or ('Genre' in b.text):
                        span = b.find_next('span')
                        links = span.find_all('a')
                        for g in links:
                            genre.append(g.text.strip())
                        break
        except:
            genre = None
            logging.warning('Не удалось найти жанры')

        try:
            try:
                price = soup0.find_all('div', {'class': 'game_purchase_price price'})[0].text.strip()
            except:
                price = soup0.find_all('div', {'class': 'discount_original_price'})[0].text.strip()
        except:
            price = None
            logging.warning('Не удалось найти цену')

        try:
            blocks = soup0.find_all('div', {'class': 'game_area_purchase_game_wrapper'})
            discount = '0%'
            for block in blocks:
                title = block.find('h2')
                title = title.text.strip()
                if (title == 'Купить ' + name) or (title == 'Buy ' + name):
                    d = block.find('div', {'class': 'discount_pct'})
                    if d:
                        discount = d.text.strip()
                    break
        except:
            discount = None
            logging.warning('Не удалось найти скидку')

        try:
            dlc = 0
            dlc_all = soup0.find('div', {'class': 'game_area_dlc_section'})
            if dlc_all:
                dlc = int(dlc_all.find_all('em')[0].text.strip().replace('(', '').replace(')', ''))
        except:
            dlc = None
            logging.warning('Не удалось найти dlc')

        try:
            tags = []
            for tag in soup0.find_all('a', {'class': 'app_tag'}):
                tags.append(tag.text.strip())   
        except:
            tags = None
            logging.warning('Не удалось найти теги')

        try:
            block = soup0.find_all('div', {'class': 'outlier_totals recent review_box_background_secondary'})
            for b in block:
                if b and b.find('span', {'class': 'game_review_summary'}):
                    recent_rating = soup0.find_all('span', {'class': 'game_review_summary'})[0].text
        except:
            recent_rating = None
            logging.warning('Не удалось найти недавний рейтинг')
        
        try:
            block = soup0.find_all('div', {'class': 'outlier_totals recent review_box_background_secondary'})
            for b in block:
                if b and b.find('span', {'class': 'game_review_summary'}):
                    recent_count = b.find('span', {'class': 'review_summary_count'}).text.strip()
        except: 
            recent_count = None
            logging.warning('Не удалось найти количество недавних отзывов')

        try:
            block = soup0.find_all('div', {'class': 'outlier_totals global review_box_background_secondary'})
            for b in block:
                if b and b.find('span', {'class': 'game_review_summary'}):
                    rating = b.find('span', {'class': 'game_review_summary'}).text.strip()
        except:
            rating = None
            logging.warning('Не удалось найти общий рейтинг')
            
        try:
            block = soup0.find_all('div', {'class': 'outlier_totals global review_box_background_secondary'})
            for b in block:
                if b and b.find('span', {'class': 'game_review_summary'}):
                    count = b.find('span', {'class': 'review_summary_count'}).text.strip()
        except:
            count = None
            logging.warning('Не удалось найти общее количество')


        logging.info('Игра обработана')

        return (url0, name, release_date, developers, genre, price, discount, dlc, tags, rating, count, recent_rating, recent_count)
    except Exception as e:
        logging.exception(f'Не удалось обработать игру: {url0} - ошибка: {e}')
        return None

In [ ]:
from time import sleep
games = []
k=0

for link in urls:   
    logging.info(f'Обработано игр {k}')
    logging.info(f'Обрабатывается ссылка: {link}')
    res = GetGame(link)
        
    if res is not None:
        games.append(res)
        logging.info('Данные успешно добавлены в список games')
    else:
        logging.warning('Функция вернула None => перезапуск driver')
        try:
            driver.quit()
        except:
            pass
        driver = driver_init()
    sleep(random.uniform(0.5, 1.2))

    k+=1
    
logging.info(f'Всего собрано игр: {len(games)}')

17:54:53 - INFO - Обработано игр 0
17:54:53 - INFO - Обрабатывается ссылка: https://store.steampowered.com/app/730/CounterStrike_2/?snr=1_7_7_230_150_1
17:54:53 - INFO - Начало обработку игры
17:54:54 - INFO - HTTP Request: GET https://store.steampowered.com/app/730/CounterStrike_2/?snr=1_7_7_230_150_1&l=english "HTTP/1.1 200 OK"
17:54:54 - INFO - Игра обработана
17:54:54 - INFO - Данные успешно добавлены в список games
17:54:55 - INFO - Обработано игр 1
17:54:55 - INFO - Обрабатывается ссылка: https://store.steampowered.com/app/3321460/Crimson_Desert/?snr=1_7_7_230_150_1
17:54:55 - INFO - Начало обработку игры
17:54:55 - INFO - HTTP Request: GET https://store.steampowered.com/app/3321460/Crimson_Desert/?snr=1_7_7_230_150_1&l=english "HTTP/1.1 302 Moved Temporarily"
17:54:55 - WARNING - Не удалось использовать client
17:54:58 - INFO - Обнаружена страница с подтверждением возраста
17:55:08 - INFO - Возраст подтвержден, страница игры открыта
17:55:08 - WARNING - Не удалось найти дату рел

In [179]:
df = pd.DataFrame(games)
df.columns = ['link', 'name', 'release_date', 'developers', 'genre', 'price', 'discount', 'dlc', 'tags', 'rating', 'reviews_count', 'recent_rating', 'recent_reviews_count']
df.head(10)

,link,name,release_date,developers,genre,price,discount,dlc,tags,rating,reviews_count,recent_rating,recent_reviews_count
0,https://store.steampowered.com/app/730/Counter...,Counter-Strike 2,2012-08-21,Valve,"[Action, Free To Play]",Free To Play,0%,1.0,"[FPS, Shooter, Multiplayer, Competitive, Actio...",Very Positive,"9,518,629",Mostly Positive,"94,966"
1,https://store.steampowered.com/app/3321460/Cri...,Crimson Desert,NaT,Pearl Abyss,"[Экшены, Приключенческие игры]","69,99€",0%,1.0,"[Открытый мир, Экшен, Для одного игрока, Прикл...",Очень положительные,77 262,None,None
2,https://store.steampowered.com/app/2868840/Sla...,Slay the Spire 2,2026-03-05,Mega Crit,"[Indie, Strategy, Early Access]","22,99€",0%,0.0,"[Strategy, Roguelike, Card Game, Deckbuilding,...",Very Positive,"97,061",Very Positive,"94,351"
3,https://store.steampowered.com/app/3764200/Res...,Resident Evil Requiem,2026-02-27,"CAPCOM Co., Ltd.","[Action, Adventure]","69,99€",0%,2.0,"[Survival Horror, Zombies, Horror, Third-Perso...",Overwhelmingly Positive,"91,878",Overwhelmingly Positive,"35,073"
4,https://store.steampowered.com/app/2050650/Res...,Resident Evil 4,2023-03-24,"CAPCOM Co., Ltd.","[Action, Adventure]","100,24€",-60%,27.0,"[Horror, Action, Survival Horror, Third-Person...",Overwhelmingly Positive,"156,587",Overwhelmingly Positive,"7,171"
5,https://store.steampowered.com/app/230410/Warf...,Warframe,2013-03-25,Digital Extremes,"[Action, RPG, Free To Play]",Free To Play,0%,17.0,"[Free to Play, Looter Shooter, Action RPG, Thi...",Very Positive,"657,387",Very Positive,"4,552"
6,https://store.steampowered.com/app/1172470/Ape...,Apex Legends™,2020-11-05,Respawn,"[Action, Adventure, Free To Play]",Free To Play,0%,0.0,"[Free to Play, Battle Royale, Multiplayer, FPS...",Mixed,"1,040,733",Mixed,"8,199"
7,https://store.steampowered.com/app/1174180/Red...,Red Dead Redemption 2,2019-12-05,Rockstar Games,"[Action, Adventure]","59,99€",-75%,0.0,"[Open World, Story Rich, Western, Multiplayer,...",Very Positive,"794,386",Very Positive,"12,143"
8,https://store.steampowered.com/app/236390/War_...,War Thunder,2013-08-15,Gaijin Entertainment,"[Action, Massively Multiplayer, Simulation, Fr...",Free To Play,0%,37.0,"[Free to Play, Simulation, Vehicular Combat, C...",Mostly Positive,"733,485",Mostly Positive,"12,608"
9,https://store.steampowered.com/app/3564740/Whe...,Where Winds Meet,2025-11-14,Everstone Studio,"[Action, Adventure, RPG, Free To Play]",Free To Play,0%,0.0,"[Open World, Free to Play, Multiplayer, Souls-...",Very Positive,"106,641",Mostly Positive,"3,361"


In [180]:
len(games)

7496

In [181]:
df.to_excel('games3.xlsx', index=False)

In [182]:
df = pd.DataFrame(games)
df.to_csv('games.csv', index=False)